In [2]:
import time
import pennylane as qml

# CPU 版本
dev_cpu = qml.device("lightning.qubit", wires=10)
# GPU 版本
dev_gpu = qml.device("lightning.gpu", wires=10)

def circuit(device):
    @qml.qnode(device)
    def ansatz():
        for i in range(10):
            qml.Hadamard(wires=i)
        return qml.expval(qml.PauliZ(0))
    return ansatz()

# 测试 CPU 时间
start = time.time()
circuit(dev_cpu)
cpu_time = time.time() - start

# 测试 GPU 时间
start = time.time()
circuit(dev_gpu)
gpu_time = time.time() - start

print(f"CPU time: {cpu_time:.4f}s, GPU time: {gpu_time:.4f}s")

CPU time: 0.0217s, GPU time: 0.3159s


In [3]:
dev = qml.device("lightning.gpu", wires=10)
print(type(dev))  # 应该输出类似 <class 'pennylane_lightning_gpu.lightning_gpu.LightningGPU'>

<class 'pennylane_lightning.lightning_gpu.lightning_gpu.LightningGPU'>


In [1]:
from pennylane.fermi import FermiSentence, from_string
import pennylane as qml
from pennylane import numpy as np
import time
qml.about()

Name: pennylane
Version: 0.43.1
Summary: PennyLane is a cross-platform Python library for quantum computing, quantum machine learning, and quantum chemistry. Train a quantum computer the same way as a neural network.
Home-page: 
Author: 
Author-email: 
License-Expression: Apache-2.0
Location: /home/lzs/.conda/envs/lzsgpu/lib/python3.12/site-packages
Requires: appdirs, autograd, autoray, cachetools, diastatic-malt, networkx, numpy, packaging, pennylane-lightning, requests, rustworkx, scipy, tomlkit, typing_extensions
Required-by: pennylane_catalyst, pennylane_lightning, pennylane_lightning_gpu

Platform info:           Linux-6.11.0-26-generic-x86_64-with-glibc2.39
Python version:          3.12.12
Numpy version:           2.3.4
Scipy version:           1.16.2
JAX version:             0.6.2
Installed devices:
- default.clifford (pennylane-0.43.1)
- default.gaussian (pennylane-0.43.1)
- default.mixed (pennylane-0.43.1)
- default.qubit (pennylane-0.43.1)
- default.qutrit (pennylane-0.43.1)


In [2]:
# 1. 导入必要库（和之前一致，必须用 scipy.sparse）
import scipy.sparse as sp

# 2. 直接写文件名（相对路径，同目录下无需加任何路径前缀）
matrix_file = "L=6 n=3.npz"  # 重点：只写文件名，不用加 /Users/... 之类的路径

# 3. 加载矩阵（一行搞定，自动恢复 CSR 格式）
H_3 = sp.load_npz(matrix_file)

# 4. 验证加载成功（可选，快速确认没出错）
print("✅ 矩阵调用成功！")
print(f"矩阵格式：{H_3.format}")  # 输出 'csr'（和保存时一致）
print(f"矩阵形状：{H_3.shape}")    # 输出原矩阵的行数×列数
print(f"非零元素个数：{H_3.nnz}")  # 输出非零元素数量

from scipy.sparse.linalg import eigsh
from scipy.linalg import eigh
H= H_3.toarray()
# 计算模最小的特征值（注意可能为复数）
min_eigval = eigsh(H, k=2, which='SA', return_eigenvectors=True,)

min =  eigh( H,eigvals_only=True)[0]
print("最小特征值:", min_eigval[0])
print(min)

✅ 矩阵调用成功！
矩阵格式：csr
矩阵形状：(309, 309)
非零元素个数：7765
最小特征值: [-8.48179737  4.02035168]
-8.481797373159006


In [3]:

def get_Hami(H):
    """
    将任意维度的哈密顿量矩阵扩展为 2^Nq 维度（量子比特系统希尔伯特空间维度）

    参数:
        H: 原哈密顿量矩阵（quspin 哈密顿量或 numpy 数组，维度为 d×d）
    返回:
        Hami: 扩展后的哈密顿量矩阵（维度 l×l，l=2^Nq）
        Nq: 所需量子比特数（满足 2^Nq ≥ d 的最小整数）
    说明:
        量子比特系统的希尔伯特空间维度必为 2 的幂次，原玻色子基矢空间维度 Ns 可能非 2^Nq，
        故通过补零扩展至最近的 2^Nq 维度，确保与量子计算框架兼容
    """
    # 若输入为 quspin 哈密顿量，先转换为 numpy 数组（优先保留稀疏性，此处暂转 dense 便于理解）
    if hasattr(H, 'toarray'):
        H_dense = H.toarray()
    else:
        H_dense = np.asarray(H)

    d = H_dense.shape[0]  # 原哈密顿量维度（即玻色子基矢空间维度 Ns）
    Nq = int(np.ceil(np.log2(d)))  # 满足 2^Nq ≥ d 的最小量子比特数
    l = 2 ** Nq  # 扩展后的维度（量子比特系统维度）

    # 初始化扩展矩阵（补零）
    Hami = np.zeros((l, l), dtype=H_dense.dtype)
    np.fill_diagonal(Hami, 100)

    Hami[:d, :d] = H_dense  # 原矩阵嵌入扩展矩阵的左上角，其余位置补零

    return Hami, Nq

Hami ,n_qubits  = get_Hami(H)
print(n_qubits)
H_decomp = qml.pauli_decompose(Hami)

9
